In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การลดความผิดพลาดด้วย Tensor Network (TEM): Qiskit Function โดย Algorithmiq
> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ใช้ได้เฉพาะผู้ใช้แผน IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลงได้


<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## ภาพรวม
วิธี Tensor-network Error Mitigation (TEM) ของ Algorithmiq เป็นอัลกอริทึมแบบ hybrid
quantum-classical ที่ออกแบบมาเพื่อทำการลดความผิดพลาดของ noise ทั้งหมดในขั้นตอน
post-processing แบบ classical ด้วย TEM ผู้ใช้สามารถคำนวณค่าความคาดหวัง
(expectation values) ของ observable ได้โดยลดข้อผิดพลาดที่เกิดจาก noise
อันหลีกเลี่ยงไม่ได้บนฮาร์ดแวร์ควอนตัม ด้วยความแม่นยำและประสิทธิภาพด้านต้นทุนที่เพิ่มขึ้น
ทำให้เป็นตัวเลือกที่น่าสนใจอย่างมากสำหรับนักวิจัยด้านควอนตัมและผู้ปฏิบัติงานในอุตสาหกรรม

วิธีการนี้ประกอบด้วยการสร้าง tensor network ที่แทนค่าผกผันของช่องทาง noise ส่วนกลาง
ที่กระทบต่อสถานะของโปรเซสเซอร์ควอนตัม แล้วนำ map นั้นไปใช้กับผลการวัดที่ครอบคลุมข้อมูล
อย่างสมบูรณ์ (informationally complete) ที่เก็บจากสถานะที่มี noise เพื่อให้ได้ตัวประมาณค่า
แบบไม่เอนเอียง (unbiased estimators) สำหรับ observable

ในฐานะข้อได้เปรียบ TEM ใช้ประโยชน์จากการวัดที่ครอบคลุมข้อมูลอย่างสมบูรณ์เพื่อเข้าถึง
observable ที่ผ่านการลด noise จำนวนมาก และมี sampling overhead บนฮาร์ดแวร์ควอนตัม
ที่เหมาะสมที่สุด ตามที่อธิบายใน Filippov et al. (2023),
[arXiv:2307.11740](https://arxiv.org/abs/2307.11740) และ Filippov et al. (2024),
[arXiv:2403.13542](https://arxiv.org/abs/2403.13542) overhead ของการวัดหมายถึงจำนวน
การวัดเพิ่มเติมที่ต้องการเพื่อทำการลด noise อย่างมีประสิทธิภาพ ซึ่งเป็นปัจจัยสำคัญ
ในความเป็นไปได้ของการคำนวณควอนตัม ดังนั้น TEM จึงมีศักยภาพในการเปิดใช้งาน
quantum advantage ในสถานการณ์ที่ซับซ้อน เช่น การประยุกต์ใช้ในสาขา quantum chaos,
many-body physics, Hubbard dynamics และการจำลองเคมีของโมเลกุลขนาดเล็ก

คุณสมบัติหลักและประโยชน์ของ TEM สรุปได้ดังนี้:

1.  **Measurement overhead ที่เหมาะสมที่สุด**: TEM เหมาะสมที่สุดเมื่อเทียบกับ
[ขอบเขตทางทฤษฎี](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601)
หมายความว่าไม่มีวิธีใดที่จะมี measurement overhead น้อยกว่านี้ได้ กล่าวอีกนัยหนึ่ง
TEM ต้องการจำนวนการวัดเพิ่มเติมน้อยที่สุดในการลด noise ซึ่งหมายความว่า TEM ใช้
quantum runtime น้อยที่สุด
2.  **ความคุ้มค่าด้านต้นทุน**: เนื่องจาก TEM จัดการการลด noise ทั้งหมดในขั้นตอน
post-processing จึงไม่จำเป็นต้องเพิ่ม circuit เพิ่มเติมบนคอมพิวเตอร์ควอนตัม
ซึ่งไม่เพียงทำให้การคำนวณถูกลงเท่านั้น แต่ยังลดความเสี่ยงในการเพิ่มข้อผิดพลาด
จากความไม่สมบูรณ์ของอุปกรณ์ควอนตัมด้วย
3.  **การประมาณค่า observable หลายตัว**: ด้วยการวัดที่ครอบคลุมข้อมูลอย่างสมบูรณ์
TEM ประมาณค่า observable หลายตัวได้อย่างมีประสิทธิภาพจากข้อมูลการวัดชุดเดียวกัน
จากคอมพิวเตอร์ควอนตัม
4.  **การลด readout error**: TEM Qiskit Function ยังมี
[วิธีการลด readout error แบบเฉพาะ](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
ที่สามารถลด readout error ได้อย่างมีนัยสำคัญหลังการ calibrate ในเวลาสั้น
5.  **ความแม่นยำ**: TEM ปรับปรุงความแม่นยำและความน่าเชื่อถือของการจำลองควอนตัม
ดิจิทัลอย่างมีนัยสำคัญ ทำให้อัลกอริทึมควอนตัมมีความแม่นยำและน่าเชื่อถือมากขึ้น
## คำอธิบาย
ฟังก์ชัน TEM ช่วยให้ได้ค่าความคาดหวังที่ผ่านการลด noise สำหรับ observable หลายตัว
บน quantum Circuit ด้วย sampling overhead น้อยที่สุด Circuit จะถูกวัดด้วย
informationally complete positive operator-valued measure (IC-POVM) และผลการวัด
ที่เก็บได้จะถูกประมวลผลบนคอมพิวเตอร์ classical การวัดนี้ใช้ในการดำเนินการวิธี
tensor network และสร้าง noise-inversion map ฟังก์ชันนำ map ที่กลับค่า noise ของ
Circuit ทั้งหมดโดยใช้ tensor network เพื่อแทนชั้นที่มี noise

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "การประมาณ observable O ที่ผ่านการลด noise ผ่านการประมวลผลผลการวัดของโปรเซสเซอร์ควอนตัมที่มี noise U และ N แทนการดำเนินการควอนตัมในอุดมคติและ noise map ที่เกี่ยวข้อง ซึ่งโดยทั่วไปอาจไม่ใช่แบบ local (และขยายเป็นกล่องสีเทา) D แทน tensor ของตัวดำเนินการที่เป็น dual กับ effect ในการวัด IC โมดูลลด noise M คือ tensor network ที่ถูก contract อย่างมีประสิทธิภาพจากกลางออกไปสองด้าน การ contraction ครั้งแรกแสดงด้วยเส้นสีม่วงแบบจุด ครั้งที่สองด้วยเส้นประ และครั้งที่สามด้วยเส้นทึบ")

เมื่อ Circuit ถูกส่งไปยังฟังก์ชัน จะถูก transpile และ optimize เพื่อลดจำนวนชั้นที่มี
two-qubit gate (gate ที่มี noise มากที่สุดบนอุปกรณ์ควอนตัม) Noise ที่กระทบต่อชั้นต่างๆ
จะถูกเรียนรู้ผ่าน
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
โดยใช้โมเดล noise แบบ sparse Pauli-Lindblad ตามที่อธิบายใน E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866)

โมเดล noise เป็นคำอธิบายที่แม่นยำของ noise บนอุปกรณ์ที่สามารถจับคุณสมบัติ
ที่ละเอียดอ่อนได้ รวมถึง qubit cross-talk อย่างไรก็ตาม noise บนอุปกรณ์อาจผันผวน
และเปลี่ยนแปลง และ noise ที่เรียนรู้ไว้อาจไม่แม่นยำ ณ เวลาที่ทำการประมาณค่า
ซึ่งอาจส่งผลให้ได้ผลลัพธ์ที่ไม่แม่นยำ
## เริ่มต้นใช้งาน
ยืนยันตัวตนด้วย [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/) และเลือกฟังก์ชัน TEM ดังนี้ (ตัวอย่างนี้สมมติว่า[บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client)ไว้ในสภาพแวดล้อม local แล้ว)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## ตัวอย่าง
ตัวอย่างต่อไปนี้แสดงการใช้ TEM เพื่อคำนวณค่าความคาดหวังของ observable จาก quantum Circuit อย่างง่าย

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

ใช้ Qiskit Serverless APIs เพื่อตรวจสอบสถานะของ Qiskit Function workload:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

ดึงผลลัพธ์ได้ดังนี้:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** ค่าความคาดหวังสำหรับ Circuit ที่ไม่มี noise สำหรับตัวดำเนินการที่กำหนดควรอยู่ที่ประมาณ `0.18409094298943401`
## อินพุต
**พารามิเตอร์**

Name | Type | Description | Required | Default | Example
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | iterable ของอ็อบเจกต์แบบ PUB (primitive unified bloc) เช่น tuple `(circuit, observables)` หรือ `(circuit, observables, parameters, precision)` ดูข้อมูลเพิ่มเติมที่ [ภาพรวมของ PUB](/guides/primitive-input-output#overview-of-pubs) หาก circuit ที่ส่งเข้ามาไม่ใช่ ISA จะถูก transpile ด้วยการตั้งค่าที่เหมาะสมที่สุด หากเป็น ISA circuit จะไม่ถูก transpile; ในกรณีนี้ observable ต้องนิยามบน QPU ทั้งหมด | ใช่ | N/A | (circuit, observables)
`backend_name` | str | ชื่อของ Backend ที่จะใช้ทำการ query| ไม่ | หากไม่ระบุ จะใช้ Backend ที่ว่างที่สุด | "ibm_fez"
`options` | dict | อ็อปชันอินพุต ดูรายละเอียดเพิ่มเติมในส่วน `Options` | ไม่ | ดูส่วน `Options` สำหรับค่าดีฟอลต์| {"max_bond_dimension": 100\}

> **Caution:** TEM มีข้อจำกัดดังต่อไปนี้ในปัจจุบัน:
> 
>   - ไม่รองรับ circuit แบบ parametrized อาร์กิวเมนต์ parameters ควรตั้งเป็น `None` หากมีการระบุ precision ข้อจำกัดนี้จะถูกลบออกในเวอร์ชันอนาคต
>   - รองรับเฉพาะ circuit ที่ไม่มี loop เท่านั้น ข้อจำกัดนี้จะถูกลบออกในเวอร์ชันอนาคต
>   - ไม่รองรับ gate ที่ไม่ใช่ unitary เช่น reset, measure และ control flow ทุกรูปแบบ การรองรับ reset จะเพิ่มในรุ่นต่อไป
### อ็อปชัน
dictionary ที่มีอ็อปชันขั้นสูงสำหรับ TEM โดย dictionary อาจมีคีย์ตามตารางด้านล่าง หากไม่ได้ระบุอ็อปชันใด จะใช้ค่าดีฟอลต์ที่แสดงในตาราง ค่าดีฟอลต์เหล่านี้เหมาะสมสำหรับการใช้งาน TEM ทั่วไป

Name | Choices | Description  | Default
-- | -- | -- | --
`tem_max_bond_dimension` | int | bond dimension สูงสุดที่จะใช้สำหรับ tensor network | 500 |
`tem_compression_cutoff` | float | ค่า cutoff ที่จะใช้สำหรับ tensor network | 1e-16
`compute_shadows_bias_from_observable` | bool | flag บูลีนที่ระบุว่าจะปรับ bias สำหรับโปรโตคอล classical shadows measurement ให้เหมาะกับ PUB observable หรือไม่ หาก False จะใช้โปรโตคอล classical shadows (ความน่าจะเป็นเท่ากันในการวัด Z, X, Y)| False
`shadows_bias` | np.ndarray | bias ที่จะใช้สำหรับโปรโตคอล randomized classical shadows measurement เป็น array 1d หรือ 2d ขนาด 3 หรือรูปร่าง (num_qubits, 3) ตามลำดับ โดยลำดับคือ ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int หรือ `None` | เวลา execution สูงสุดบน QPU เป็นวินาที หากระยะเวลา runtime เกินค่านี้ job จะถูกยกเลิก หาก `None` จะใช้ขีดจำกัดดีฟอลต์ที่กำหนดโดย Qiskit Runtime | `None`
`num_randomizations` | int | จำนวนการ randomize ที่จะใช้สำหรับการเรียนรู้ noise และ gate twirling | 32
`max_layers_to_learn` | int | จำนวนชั้นที่ไม่ซ้ำสูงสุดที่จะเรียนรู้ | 4
`mitigate_readout_error` | bool | flag บูลีนที่ระบุว่าจะทำการลด readout error หรือไม่ | True
`num_readout_calibration_shots` | int | จำนวน shot ที่จะใช้สำหรับการลด readout error | 10000
`default_precision` | float | ค่า precision ดีฟอลต์ที่จะใช้สำหรับ PUB ที่ไม่ได้ระบุ precision | 0.02
`seed` | int หรือ `None` | กำหนด seed ของตัวสร้างตัวเลขสุ่มเพื่อความสามารถในการทำซ้ำ หาก `None` จะไม่กำหนด seed | None
## เอาต์พุต
[PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) ของ Qiskit ที่มีผลลัพธ์ที่ผ่านการลด noise ด้วย TEM ผลลัพธ์สำหรับแต่ละ PUB จะถูกส่งคืนเป็น [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) ที่มีฟิลด์ดังต่อไปนี้:

Name |Type | Description
-- | -- | --
data | DataBin | [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) ของ Qiskit ที่มีค่า observable และ standard error ที่ผ่านการลด noise ด้วย TEM โดย DataBin มีฟิลด์ดังนี้: <ul><li>`evs`: ค่า observable ที่ผ่านการลด noise ด้วย TEM</li><li>`stds`: standard error ของ observable ที่ผ่านการลด noise ด้วย TEM</li></ul>
metadata | dict | dictionary ที่มีผลลัพธ์เพิ่มเติม โดย dictionary มีคีย์ดังต่อไปนี้: <ul><li>`"evs_non_mitigated"`: ค่า observable โดยไม่มีการลด noise</li><li>`"stds_non_mitigated"`: standard error ของผลลัพธ์โดยไม่มีการลด noise</li><li>`"evs_mitigated_no_readout_mitigation"`: ค่า observable ที่มีการลด noise แต่ไม่มีการลด readout error</li><li>`"stds_mitigated_no_readout_mitigation"`: standard error ของผลลัพธ์ที่มีการลด noise แต่ไม่มีการลด readout error</li><li>`"evs_non_mitigated_with_readout_mitigation"`: ค่า observable โดยไม่มีการลด noise แต่มีการลด readout error</li><li>`"stds_non_mitigated_with_readout_mitigation"`: standard error ของผลลัพธ์โดยไม่มีการลด noise แต่มีการลด readout error</li></ul>
## การดึงข้อความแจ้งข้อผิดพลาด
หากสถานะ workload เป็น ERROR ให้ใช้ job.result() เพื่อดึงข้อความแจ้งข้อผิดพลาดดังนี้: